# Phân loại bất thường tim mạch từ ECG 12 chuyển đạo (PTB-XL): notebook chạy toàn bộ nhóm trên Google Colab

Một lần chạy từ trên xuống thực hiện phần việc của **cả ba thành viên**. Mỗi phần có nhãn rõ ràng:

| Nhãn | Thành viên | Phần việc trong notebook |
|---|---|---|
| **[A]** | Nguyễn Mạnh Hiếu | dữ liệu, thống kê mô tả, baseline LR/RF, tổng hợp kết quả |
| **[B]** | Nguyễn Khắc Tiếp | huấn luyện `cnn_simple`, `resnet1d`, `cnn_attn` |
| **[C]** | Trần Mạnh Hùng | huấn luyện `gru`, `lstm`, ablation (số chuyển đạo, augmentation), saliency |

Khi chạy, mỗi bước in một dòng đầu `[A]`, `[B]` hoặc `[C]` kèm tên thành viên. Log của từng người ghi riêng vào `experiments_A.csv`, `experiments_B.csv`, `experiments_C.csv`, nên luôn biết lượt chạy nào thuộc ai.
Phần cuối in bảng kết quả **riêng cho từng thành viên** và bảng tổng hợp chung.

**Cách dùng**
1. `Runtime` → `Change runtime type` → GPU (T4).
2. Chạy lần lượt từ trên xuống. Lần đầu nên đặt `EPOCHS = 2` ở ô Cấu hình để thử nhanh toàn bộ quy trình, rồi đặt lại `30` để chạy thật.
3. Kết quả và log ghi vào Google Drive nên không mất khi Colab ngắt. Chạy lại ô huấn luyện sẽ bỏ qua các lượt đã xong.
4. Muốn chỉ chạy phần của một vài người: đặt `RUN["B"] = False`... ở ô Cấu hình.

> Notebook chỉ chạy trọn vẹn khi các phần TODO trong `ptbxl_ecg/` đã được cài đặt (ô kiểm tra test cho biết phần nào còn thiếu).

## 0. Cấu hình

In [ ]:
# ====== CẦN SỬA (nếu muốn) ======
REPO_URL = "https://github.com/TiepKun/ptbxl-ecg.git"   # repo công khai của nhóm
BRANCH = "main"

MEMBERS = {"A": "Nguyễn Mạnh Hiếu", "B": "Nguyễn Khắc Tiếp", "C": "Trần Mạnh Hùng"}
RUN = {"A": True, "B": True, "C": True}      # đặt False để bỏ qua phần của người nào đó
MODELS = {"B": ["cnn_simple", "resnet1d", "cnn_attn"],   # [B] các mô hình CNN / attention
          "C": ["gru", "lstm"]}                          # [C] các mô hình RNN
BEST_MODEL = None     # None: tự chọn theo val AUC trung bình; hoặc gõ tên, ví dụ "resnet1d" (dùng cho ablation của C)

SEEDS = [0, 1, 2]     # yêu cầu: ít nhất 3 seed cho mỗi cấu hình
EPOCHS = 30           # lần thử đầu: 2
BATCH_SIZE = 128
# ================================

import subprocess

def sh(cmd):
    """Chạy lệnh shell và in kết quả ra ngay khi có."""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def banner(k, text):
    """In dòng đầu cho biết phần này của thành viên nào."""
    print("=" * 78)
    print(f"[{k}] {MEMBERS[k]}  |  {text}")
    print("=" * 78)

SEEDS_STR = " ".join(map(str, SEEDS))

## 1. Môi trường: GPU, Google Drive, mã nguồn

In [ ]:
sh("nvidia-smi -L")
import torch
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/ptbxl-ecg"          # file đã xử lý, runs và log được lưu ở đây
RAW = "/content/ptbxl_raw"                          # dữ liệu thô để trên đĩa Colab (nhanh hơn Drive nhiều)
PROC, RUNS = f"{DRIVE}/processed", f"{DRIVE}/runs"
LOG_DIR = f"{DRIVE}/logs"
LOG_CSV = {k: f"{LOG_DIR}/experiments_{k}.csv" for k in MEMBERS}   # mỗi người một file log
for d in (RAW, PROC, RUNS, LOG_DIR):
    os.makedirs(d, exist_ok=True)
print("Thư mục Drive:", DRIVE)

In [ ]:
if not os.path.exists("/content/ptbxl-ecg"):
    sh(f"git clone -b {BRANCH} {REPO_URL} /content/ptbxl-ecg")
os.chdir("/content/ptbxl-ecg")
sh("git pull --ff-only")
# Colab đã có sẵn torch; chỉ cài các gói còn lại
sh("grep -v '^torch' requirements.txt > /tmp/req.txt && pip install -q -r /tmp/req.txt")
sh("git log -1 --oneline")

## 2. Kiểm tra mã trước khi chạy dữ liệu thật

Mỗi người có bộ test riêng; `NotImplementedError: TODO [X]` nghĩa là phần của người đó chưa được cài đặt.
Test dùng dữ liệu giả lập nên chạy trong vài chục giây.

In [ ]:
TESTS = {"A": "tests/test_A_data_eval.py", "B": "tests/test_B_cnn_train.py", "C": "tests/test_C_rnn_ablation.py"}
test_status = {}
for k, path in TESTS.items():
    banner(k, f"kiểm tra {path}")
    test_status[k] = sh(f"python -m pytest -q {path} --no-header -p no:cacheprovider --tb=line 2>&1 | tail -8")
print("\nLưu ý: nếu phần nào của ai còn lỗi NotImplementedError thì phần huấn luyện tương ứng bên dưới sẽ lỗi.")

## 3. Dữ liệu PTB-XL  **[A] Nguyễn Mạnh Hiếu**

In [ ]:
banner("A", "tải và xử lý dữ liệu (dùng chung cho cả nhóm)")
if not os.path.exists(f"{PROC}/X.npy"):
    if not os.path.exists(f"{RAW}/ptbxl_database.csv"):
        # Tải bản 100 Hz + 2 file CSV từ kho AWS công khai của PhysioNet (song song, nhanh); bỏ qua bản 500 Hz
        sh("pip install -q awscli")
        sh(f'aws s3 sync --no-sign-request s3://physionet-open/ptb-xl/1.0.3/ {RAW} --exclude "records500/*" --only-show-errors')
    sh(f'echo "Số file .hea:" $(find {RAW}/records100 -name "*.hea" | wc -l)')
    # Đọc tín hiệu, ánh xạ nhãn sang 5 siêu lớp, lưu X.npy / Y.npy / folds.npy vào Drive
    sh(f"python -m ptbxl_ecg.prepare_data --raw-dir {RAW} --out-dir {PROC}")
else:
    print("Đã có dữ liệu đã xử lý tại", PROC, "(bỏ qua tải và xử lý)")

In [ ]:
banner("A", "thống kê mô tả (số bản ghi, phân bố nhãn theo fold) -> results/")
sh(f"python -m ptbxl_ecg.eda --data-dir {PROC} --out-dir results")

## 4. Mô hình cơ sở: đặc trưng thủ công + LR / RF  **[A] Nguyễn Mạnh Hiếu**

In [ ]:
if RUN["A"]:
    banner("A", "baseline Logistic Regression và Random Forest")
    sh(f"python -m ptbxl_ecg.baseline --data-dir {PROC} --log-csv {LOG_CSV['A']} --seeds {SEEDS_STR}")
else:
    print("[A] bỏ qua (RUN['A'] = False)")

## 5. So sánh kiến trúc (nhóm `main`)

Chạy lại sau khi Colab ngắt sẽ tự bỏ qua các lượt đã xong.

### 5a. **[B] Nguyễn Khắc Tiếp**: CNN, ResNet 1D, CNN + attention

In [ ]:
if RUN["B"]:
    banner("B", "huấn luyện " + ", ".join(MODELS["B"]))
    sh(f"python -m ptbxl_ecg.ablation --group main --models {' '.join(MODELS['B'])} --seeds {SEEDS_STR} "
       f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV['B']}")
else:
    print("[B] bỏ qua (RUN['B'] = False)")

### 5b. **[C] Trần Mạnh Hùng**: BiGRU, BiLSTM

In [ ]:
if RUN["C"]:
    banner("C", "huấn luyện " + ", ".join(MODELS["C"]))
    sh(f"python -m ptbxl_ecg.ablation --group main --models {' '.join(MODELS['C'])} --seeds {SEEDS_STR} "
       f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV['C']}")
else:
    print("[C] bỏ qua (RUN['C'] = False)")

## 6. Ablation: số chuyển đạo và augmentation  **[C] Trần Mạnh Hùng**

Chạy trên **một** mô hình, chọn theo **val AUC** (không dùng test để chọn). `cnn_simple` (baseline học sâu) không được chọn.

In [ ]:
import glob, pandas as pd
def load_logs():
    files = sorted(set(glob.glob(f"{LOG_DIR}/experiments_*.csv")))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame()

if RUN["C"]:
    df = load_logs()
    main_runs = df[(df["group"] == "main") & (df["model"] != "cnn_simple")]
    ranking = main_runs.groupby("model")["val_auc"].agg(["mean", "count"]).sort_values("mean", ascending=False)
    display(ranking)
    if BEST_MODEL is None:
        BEST_MODEL = ranking.index[0]
    banner("C", f"mô hình dùng cho ablation: {BEST_MODEL}")

In [ ]:
if RUN["C"]:
    banner("C", f"ablation số chuyển đạo (limb6, II) với {BEST_MODEL}")
    sh(f"python -m ptbxl_ecg.ablation --group leads --model {BEST_MODEL} --seeds {SEEDS_STR} "
       f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV['C']}")
    banner("C", f"ablation tắt augmentation với {BEST_MODEL}")
    sh(f"python -m ptbxl_ecg.ablation --group aug --model {BEST_MODEL} --seeds {SEEDS_STR} "
       f"--epochs {EPOCHS} --batch-size {BATCH_SIZE} --data-dir {PROC} --out-dir {RUNS} --log-csv {LOG_CSV['C']}")
else:
    print("[C] bỏ qua (RUN['C'] = False)")

## 7. Kết quả theo từng thành viên

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display

def owner(row):
    """Gán mỗi lượt chạy cho người phụ trách: baseline -> A; main -> B hoặc C theo mô hình; leads/aug -> C."""
    if row["group"] == "baseline":
        return "A"
    if row["group"] == "main":
        return "B" if row["model"] in MODELS["B"] else "C"
    return "C"

df = load_logs()
assert len(df), "Chưa có log nào trong " + LOG_DIR
df = df[df["group"] != "dev"].drop_duplicates("run_name", keep="last")
df["member"] = df.apply(owner, axis=1)
os.makedirs("logs", exist_ok=True)
df.drop(columns="member").to_csv("logs/experiments.csv", index=False)     # log gộp để nộp

AUC_COLS = [c for c in df.columns if c.startswith("auc_")]
def table(sub):
    g = sub.groupby(["group", "model", "leads", "aug"])
    out = g.agg(n_seeds=("seed", "count"), params=("params", "first"),
                test_auc_mean=("test_auc", "mean"), test_auc_std=("test_auc", lambda x: x.std(ddof=1) if len(x) > 1 else 0.0),
                val_auc_mean=("val_auc", "mean"), test_f1_macro=("test_f1_macro", "mean"),
                **{c: (c, "mean") for c in AUC_COLS})
    return out.sort_values("test_auc_mean", ascending=False).round(4)

for k in ("A", "B", "C"):
    banner(k, "kết quả (trung bình theo seed, test fold 10)")
    sub = df[df["member"] == k]
    if sub.empty:
        print("Chưa có kết quả nào của phần này.")
    else:
        print(f"{len(sub)} lượt chạy")
        display(table(sub))

In [ ]:
# Bảng tổng hợp chung của cả nhóm (mean ± std theo seed) và biểu đồ  [A] Nguyễn Mạnh Hiếu
banner("A", "tổng hợp kết quả cả nhóm -> results/summary.md, results/test_auc.png")
sh("python -m ptbxl_ecg.aggregate --log-csv logs/experiments.csv --out-dir results")
from IPython.display import Image, Markdown
display(Markdown(open("results/summary.md", encoding="utf-8").read()))
display(Image("results/test_auc.png"))

## 8. Phân tích độ quan trọng theo chuyển đạo (saliency)  **[C] Trần Mạnh Hùng**

In [ ]:
if RUN["C"]:
    banner("C", f"saliency của {BEST_MODEL}, seed 0")
    sh(f"python -m ptbxl_ecg.saliency --run-dir {RUNS}/main_{BEST_MODEL}_12_aug1_s0 --data-dir {PROC} --out-dir results")
    for f in sorted(glob.glob("results/saliency_*.png")):
        display(Image(f))

## 9. Lưu kết quả

Chép log, kết quả ra Drive và tải về để đưa vào repo (`git add logs/ results/`).
Số liệu trong báo cáo phải khớp với `logs/experiments.csv`.

In [ ]:
import shutil
for k in MEMBERS:
    if os.path.exists(LOG_CSV[k]):
        shutil.copy(LOG_CSV[k], f"logs/experiments_{k}.csv")
shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
shutil.make_archive(f"{DRIVE}/ket_qua_nhom", "zip", ".", "logs")
shutil.make_archive(f"{DRIVE}/ket_qua_hinh", "zip", ".", "results")
print("Đã lưu vào", DRIVE)
from google.colab import files
files.download(f"{DRIVE}/ket_qua_nhom.zip")